In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import requests, zipfile, io, os
import pandas as pd

In [ ]:
# Listings fetch
domain = "datasets.techmatrix.it/airml"
token = "DI_xeno_2026"

cities = ["sicilia", "trentino", "venezia", "roma", "puglia",
          "napoli", "firenze", "milano", "bergamo", "bologna"]

for city in cities:
    data_dir = os.path.join("./data", city)
    if os.path.exists(data_dir):
        print(f"Skipping {city}: folder already exists")
        continue
    
    url = f"https://{domain}/listings/{city}.zip?token={token}"
    resp = requests.get(url, stream=True)
    if resp.ok:
        os.makedirs(data_dir, exist_ok=True)
        zip_path = os.path.join("./data", f"{city}.zip")
        with open(zip_path, "wb") as f:
            for chunk in resp.iter_content(8192):
                if chunk:
                    f.write(chunk)
        with zipfile.ZipFile(zip_path, "r") as z:
            z.extractall(path=data_dir)
        os.remove(zip_path)
    else:
        print(f"Failed to download {city}: {resp.status_code}")

In [ ]:
# # Reviews fetch

# for city in cities:
#     url = f"https://{domain}/reviews/{city}.zip?token={token}"
#     resp = requests.get(url, stream=True)
#     if resp.ok:
#         data_dir = os.path.join("./data", city)
#         if not os.path.exists(data_dir):
#             os.makedirs(data_dir, exist_ok=True)
#         zip_path = os.path.join("./data", f"{city}.zip")
#         with open(zip_path, "wb") as f:
#             for chunk in resp.iter_content(8192):
#                 if chunk:
#                     f.write(chunk)
#         with zipfile.ZipFile(zip_path, "r") as z:
#             z.extractall(path=data_dir)
#         os.remove(zip_path)
#     else:
#         print(f"Failed to download {city}: {resp.status_code}")

# Filtering

In [ ]:
COLS_TO_DROP = {
    # URL / immagini
    "listing_url", "picture_url", "host_thumbnail_url", "host_picture_url", "host_url",
    # Testuali / identificativi listing
    "name", "description", "neighborhood_overview", "calendar_updated",
    # Identificatori di scraping / metadati tecnici
    "scrape_id", "last_scraped", "source",
    # Identificatori personali / dati host
    "host_id", "host_name", "host_since", "host_location", "host_about",
    "host_neighbourhood", "host_listings_count", "host_total_listings_count",
    "host_verifications", "host_has_profile_pic", "host_identity_verified",
    # Metriche risposta host
    "host_response_time", "host_response_rate", "host_acceptance_rate", "host_is_superhost",
    # Location duplicate / non predittive
    "neighbourhood", "neighbourhood_group_cleansed",
    # Testo derivabile / calcolato
    "bathrooms_text", "first_review", "last_review",
    # Calcolati host (aggregati)
    "calculated_host_listings_count", "calculated_host_listings_count_entire_homes",
    "calculated_host_listings_count_private_rooms", "calculated_host_listings_count_shared_rooms",
}
dfs = []

for sub in os.listdir("./data"):
    subpath = os.path.join("./data", sub)
    if not os.path.isdir(subpath):
        continue

    csv_path = os.path.join(subpath, "listings.csv")

    if os.path.exists(csv_path):
        city_df = pd.read_csv(csv_path, usecols=lambda col: col not in COLS_TO_DROP)
        city_df["city"] = sub
        dfs.append(city_df)
    else:
        for root, _, files in os.walk(subpath):
            if "listings.csv" in files:
                city_df = pd.read_csv(os.path.join(root, "listings.csv"), usecols=lambda col: col not in COLS_TO_DROP)
                city_df["city"] = sub
                dfs.append(city_df)
                break

if dfs:
    listings = pd.concat(dfs, ignore_index=True)
else:
    listings = pd.DataFrame()

listings.info()

In [ ]:
# REVIEW_COLS_TO_DROP = {
#     "reviewer_name", "date"
# }
# review_dfs = []

# for sub in os.listdir("./data"):
#     subpath = os.path.join("./data", sub)
#     if not os.path.isdir(subpath):
#         continue
#     csv_path = os.path.join(subpath, "reviews.csv")
#     if os.path.exists(csv_path):
#         review_dfs.append(pd.read_csv(csv_path, usecols=lambda col: col not in REVIEW_COLS_TO_DROP))
#     else:
#         for root, _, files in os.walk(subpath):
#             if "reviews.csv" in files:
#                 review_dfs.append(pd.read_csv(os.path.join(root, "reviews.csv"), usecols=lambda col: col not in REVIEW_COLS_TO_DROP))
#                 break

# if review_dfs:
#     reviews = pd.concat(review_dfs, ignore_index=True)
# else:
#     reviews = pd.DataFrame()

# reviews.info()

In [ ]:
# 1. Drop righe duplicate
dupes = listings.duplicated().sum()
listings = listings.drop_duplicates().reset_index(drop=True)
print(f"Righe duplicate rimosse: {dupes}")

In [ ]:

# 2. Drop righe con target nullo (price) o con più del 70% di valori nulli
null_price = listings["price"].isna().sum()
listings = listings.dropna(subset=["price"])
print(f"Righe con price nullo rimosse: {null_price}")

# Righe con più del 70% di valori nulli
thresh = int(0.70 * listings.shape[1])
sparse_mask = listings.isna().sum(axis=1) > thresh
sparse_count = sparse_mask.sum()
listings = listings[~sparse_mask].reset_index(drop=True)
print(f"Righe con >70% nulli rimosse: {sparse_count}")

In [ ]:
# 3. Drop righe con accommodates < 1
low_acc = (listings["accommodates"] < 1).sum()
listings = listings[listings["accommodates"] >= 1].reset_index(drop=True)
print(f"Righe con accommodates < 1 rimosse: {low_acc}")

### SAMPLE DEI DATI

Siccome i dati sono molti. Effettuiamo un campionamento casuale per velocizzare le operazioni di preprocessing e modellazione.

In [ ]:
listings_all = listings.copy()

In [ ]:
# Eseguire questa cella per avere tutti i dati
listings = listings_all.copy()

In [ ]:
listings = listings.sample(n=50000, random_state=7112004).reset_index(drop=True)

## 2. Analisi Esplorativa dei Dati

### 2.1 Statistiche Generali


Questa sottosezione calcola le statistiche descrittive per le colonne del dataframe `listings`.

Le celle sono suddivise in sotto-sezioni:

- **2.1.a** Esplorazione delle colonne.
- **2.1.b** Statistiche descrittive: media, std, quartili, conteggio valori mancanti e unici.
- **2.1.c** Top valori per alcune colonne categoriche.
- **2.1.d** Matrice di correlazione tra variabili numeriche e relativa heatmap.


#### 2.1.a Esplorazione delle colonne

Estraiamo le info relative alle colonne del dataframe `listings`.

In [ ]:
listings.info()

Il dataset contiene 43 variabili totali. Guardiamo il contenuto di queste variabili per capire meglio la distribuzione dei dati e identificare eventuali anomalie o valori mancanti.

In [ ]:
listings.head()

Estraiamo l'elenco delle colonne numeriche presenti nel dataset `listings` per identificare quali feature numeriche sono disponibili per l'analisi. Questo ci aiuterà a capire meglio la struttura dei dati e a pianificare le fasi successive dell'analisi esplorativa.

In [ ]:
listings.select_dtypes(include=[np.number]).info()

Il dataset presenta 33 colonne numeriche. Ecco la spiegazione delle principali colonne numeriche mostrate sopra:

- **accommodates**: numero massimo di ospiti supportati.
- **bathrooms**: numero di bagni.
- **bedrooms**: numero di camere da letto.
- **beds**: numero di letti disponibili.
- **minimum_nights / maximum_nights**: vincoli min/max di soggiorno.
- **availability_30 / availability_60 / availability_90 / availability_365**: giorni disponibili nei rispettivi intervalli.
- **number_of_reviews, reviews_per_month**: conteggio recensioni e frequenza mensile.
- **review_scores_rating**: punteggio medio delle recensioni (aggregato).
- **estimated_occupancy_l365d**: occupazione stimata su ultimi 365 giorni (target Task B).
- **estimated_revenue_l365d**: ricavo stimato annuo (se presente).
- **latitude / longitude**: coordinate geografiche (possono servire per mappe o distanza dal centro).
- **n_amenities**: (se calcolata) numero di servizi offerti dall'alloggio.

Le colonne non numeriche includono variabili categoriche (es. `neighbourhood_group`, `room_type`) e testuali (es. `name`, `description`), che richiederanno approcci di analisi e preprocessing differenti, che saranno affrontati nelle sezioni successive.

#### 2.1.b Statistiche descrittive numeriche

Applico il metodo describe a listing per avere una prima idea della distribuzione dei dati.

In [ ]:
listings.describe()

Questo metodo fornisce le seguenti statistiche per ogni colonna numerica:
- **count**: numero di osservazioni non-nulle.
- **mean / std**: media e deviazione standard; confrontale per capire dispersione e presenza di outlier.
- **min / 25% / 50% / 75% / max**: quantili utili per individuare asimmetrie e outlier (max >> 75% + IQR indica outlier).

Guardiamo ora quanti valori unici e quanti valori mancanti ci sono per ogni colonna numerica. Questo ci aiuterà a capire se ci sono variabili con molti valori mancanti.

In [ ]:
stats = pd.DataFrame()
stats['missing'] = listings.isna().sum()
stats['unique'] = listings.nunique()
display(stats)

Come si può vedere molte variabili hanno un numero elevato di valori mancanti, in particolare quelle relative alle recensioni. La maggior parte però ha un numero di valori mancanti molto basso, quindi possiamo considerare di convertire i valori mancanti o di rimuovere le righe con valori mancanti a seconda del caso specifico.

#### 2.1.c Categorie top values


In questa sezione esploriamo le colonne categoriche per identificare le categorie più frequenti. In particolare guardiamo le seguenti colonne:
- `neighbourhood_cleansed`: quartiere in cui si trova l'alloggio.
- `property_type`: tipo di proprietà (es. appartamento, casa, bed & breakfast).
- `room_type`: tipo di alloggio (es. intero appartamento, stanza privata).
- `amenities`: servizi offerti (es. Wi-Fi, cucina, aria condizionata).

Cominciamo estraendo delle statistiche relative a queste colonne categoriche per identificare le categorie più frequenti e capire meglio la distribuzione dei dati in queste variabili. Questo ci aiuterà a pianificare il preprocessing e a identificare categorie che potrebbero essere raggruppate o trattate in modo speciale.

In [ ]:
neighbourhoods = listings["neighbourhood_cleansed"].value_counts()
property_types = listings["property_type"].value_counts()
room_types = listings["room_type"].value_counts()
amenities = listings["amenities"].value_counts()

In [ ]:
print(neighbourhoods.head())
neighbourhoods.describe()

In [ ]:
print(property_types.head())
print(property_types.describe())

In [ ]:
print(room_types.head())
print(room_types.describe())

In [ ]:
print(amenities.head())
amenities.describe()

Come si può vedere, la variabile `neighbourhood_cleansed` ha 1011 valori unici, `property_type` ha 118 valori unici, `room_type` ha 4 valori unici e `amenities` ha 183716 valori unici.

`property_type` e `room_type` hanno un numero di categorie alto, quindi verranno presi i top 50 più frequenti, mentre per `room_type` che ha solo 4 categorie le visualizzeremo tutte.

per quanto riguarda `amenities`, è una variabile molto complessa, in quanto contiene una lista di servizi per record. Per questo motivo, è necessario analizzare questa variabile in modo diverso rispetto alle altre variabili categoriche. Estraiamo tutti i servizi presenti nel dataset e contiamo la frequenza di ogni servizio.

Proviamo ora a visualizzare la distribuzione di queste variabili categoriche tramite dei grafici a barre, per capire se ci sono delle categorie molto frequenti che potrebbero essere utili per il modello di regressione.

Adesso andremo a visualizzare la distribuzione di `amenities`. Per fare questo, siccome i valori in `amenities` sono una stringa che rappresenta una lista di stringhe, è necessario:
1. trasformare la stringa in una lista di stringhe
2. esplodere la lista in modo da avere una riga per ogni serivizio
3. contare la frequenza di ogni servizio

In questo modo è possibile ottenere un risultato simile a quello delle altre variabili categoriche, con la frequenza di ogni categoria presente nel dataset.

In [ ]:
import ast
all_amenities = listings["amenities"].apply(ast.literal_eval).explode().value_counts()
all_amenities

Anche in questo caso, avendo 15206 servizi unici, si visualizzeranno solo i 50 più frequenti, per avere un'idea della distribuzione dei servizi presenti nel dataset.

In [ ]:
top_amenities = all_amenities.head(50)
top_amenities.plot.bar(figsize=(14, 6), title="Top 50 Amenities")

Si può notare come le categorie hanno una distribuzione piuttosto uniforme, con nessuna categoria che rappresenta una percentuale eccessiva rispetto alle altre. Tuttavia è da considerare che questo campo contiene circa 16000 categorie uniche, quindi è possibile che alcune categorie siano rappresentate da un numero molto basso di istanze.

Nella parte del pre-processing, è possibile decidere di mantenere solo i servizi più frequenti, in modo da ridurre la dimensionalità del dataset e migliorare la performance del modello di regressione. Un altro dato utile potrebbe essere contare il numero di amenità presenti in ogni record, in modo da avere una variabile numerica che rappresenta la quantità di servizi offerti da ogni alloggio, che potrebbe essere ulteriormente utile per la predizione del prezzo.

In [ ]:
top_neighbourhoods = neighbourhoods.head(50)
top_neighbourhoods.plot.bar(figsize=(14, 6), title="Top 50 Neighbourhoods")

Per quanto riguarda i quartieri, le categorie più frequenti sono i centri storici delle città più grandi, in quanto sono le zone più turistiche e quindi più richieste per l'affitto di case vacanze. Oltre ai centri storici, sono presenti anche quartieri residenziali e quartieri periferici, che potrebbero essere meno richiesti ma comunque presenti nel dataset. I livelli di granularità dei quartieri sono diversi, con alcuni quartieri che rappresentano intere città, mentre altri rappresentano solo dei quartieri specifici all'interno di una città.

In [ ]:
top_property_types = property_types.head(50)
top_property_types.plot.bar(figsize=(14, 6), title="Top 50 Property Types")

Si può vedere come le categorie più frequenti di `property_type` siano concentrate su appartamenti, case vacanze e case indipendenti, che sono le tipologie di alloggio più richieste per l'affitto di case vacanze. Oltre a queste tipologie, sono presenti anche altre tipologie di alloggio meno richieste ma comunque presenti nel dataset.

In [ ]:
room_types.plot.bar(log=True)

Questo grafico mostra la distribuzione delle tipologie di alloggio presenti nel dataset. Come si può vedere, le tipologie più frequenti sono gli appartamenti, seguiti dalle stanze private. Il grafico è in scala logaritmica, in quanto ci sono solo un migliario di stanze d'hotel o stanze condivise, mentre per le altre tipologie di alloggio sono presenti decine di migliaia di istanze.

#### 2.1.d Correlation matrix and heatmap


In [ ]:
num_cols = listings.select_dtypes(include=[np.number]).columns.tolist()
corr = listings[num_cols].corr()
plt.figure(figsize=(10,8))
sns.heatmap(corr, cmap='coolwarm', center=0, vmin=-1, vmax=1)
plt.title('Correlation matrix (numeric features)')
plt.show()

Questa sottosezione visualizza la **matrice di correlazione** tra tutte le variabili numeriche del dataframe tramite una **heatmap**:

- Ogni cella della matrice misura la correlazione lineare tra due variabili numeriche.
- I valori vanno da **-1** a **1**:
  - **1** = correlazione positiva perfetta.
  - **0** = nessuna relazione lineare evidente.
  - **-1** = correlazione negativa perfetta.
- La diagonale principale è sempre pari a 1, perché ogni variabile è perfettamente correlata con sé stessa.

Come interpretare i colori della heatmap:

- Toni **rossi**: relazione positiva, cioè le due variabili tendono a crescere insieme.
- Toni **blu**: relazione negativa, cioè una cresce mentre l'altra tende a scendere.
- Toni molto chiari o quasi neutri: relazione debole o assente.

Cose interessanti da osservare nel nostro caso:

- Blocchi forti tra `accommodates`, `bathrooms`, `bedrooms` e `beds`: indicano che descrivono dimensioni simili dell'alloggio.
- Correlazioni tra `availability_30`, `availability_60`, `availability_90` e `availability_365`: sono attese perché misurano la disponibilità nello stesso senso ma su finestre diverse.
- Relazioni tra `number_of_reviews`, `reviews_per_month` e i punteggi recensione: utili per capire se l'attività dell'alloggio è associata alla qualità percepita.
- Eventuali correlazioni con `price` e `estimated_occupancy_l365d`: sono quelle più utili in vista dei modelli di regressione.

### 2.3 Specifiche per le singole task


Questa sezione contiene le specifiche per l'analisi esplorativa dei dati mirata alle singole task.

Le celle sono suddivise in sotto-sezioni:

- **2.3.a** Price Prediction
- **2.3.b** Occupancy Regression
- **2.3.c** NLP Classification
- **2.3.d** Recommendation System


#### 2.3.a - Price Regression

Per il task di regressione su `price` è necessario filtrare i dati per rimuovere le feature non necessarie. In particolare è necessario rimuovere le feature riguardanti il tasso di occupazione, in quanto non sono importanti per la predizione del prezzo.

Creiamo quindi un nuovo DataFrame `lst_for_price_analysis` che contiene solo le feature necessarie per la predizione del prezzo. In particolare, rimuoviamo le feature riguardanti il tasso di occupazione e la disponibilità:

- `minimum_nights`, `maximum_nights`, `minimum_minimum_nights`, `maximum_minimum_nights`, `minimum_maximum_nights`, `maximum_maximum_nights`, `minimum_nights_avg_ntm`, `maximum_nights_avg_ntm`
- `calendar_updated`, `has_availability`, `availability_30`, `availability_60`, `availability_90`, `availability_365`, `calendar_last_scraped`
- `number_of_reviews`, `number_of_reviews_ltm`, `number_of_reviews_l30d`, `availability_eoy`, `number_of_reviews_ly`
- `estimated_occupancy_l365d`, `estimated_revenue_l365d`


Manteniamo quindi solo le feature che potrebbero essere utili per la predizione del prezzo, ovvero:
- `id`
- `listing_url`
- `neighbourhood_cleansed`
- `latitude`
- `longitude`
- `property_type`
- `room_type`
- `accommodates`
- `bathrooms`
- `bedrooms`
- `beds`
- `amenities`
- `price`

In [ ]:
filtered_features = [
    "id",
    "neighbourhood_cleansed",
    "latitude",
    "longitude",
    "property_type",
    "room_type",
    "accommodates",
    "bathrooms",
    "bedrooms",
    "beds",
    "amenities",
    "price"
]

lst_for_price_analysis = listings[filtered_features]

lst_for_price_analysis.info()

Il nuovo DataFrame `lst_for_price_analysis` contiene quindi 12 variabili, di cui 11 sono feature e 1 è la variabile da predire. Le feature sono sono perlopiù numeriche. Ci sono alcune feature categoriche:
- `neighbourhood_cleansed`
- `property_type`
- `room_type`
- `amenities`

Che possono essere importanti per la predizione del prezzo, ma che vanno trattate in modo diverso dalle feature numeriche. In particolare, è necessario trasformare le feature categoriche viste nella sezione precedente in nuove variabili binarie.
Inoltre bisogna trattare anche il valore di `price`, che è una stringa. Per poter utilizzare questa variabile e poterla visualizzare nei plot, è necessario trasformarla in un valore numerico.

Vediamo ora il formato della variabile `price`.

In [ ]:
print(lst_for_price_analysis["price"])

Come si può vedere, la variabile `price` è una stringa che contiene il simbolo del dollaro e le virgole per le migliaia. Per poter utilizzare questa variabile e poterla visualizzare nei plot, è necessario trasformarla in un valore numerico:

In [ ]:
lst_for_price_analysis["price"] = (
    lst_for_price_analysis["price"]
        .str.replace('$', '', regex=False)
        .str.replace(',', '', regex=False)
        .astype(float)
)
lst_for_price_analysis["price"]

Adesso `price` è una variabile numerica che può essere utilizzata per la predizione del prezzo.

Ora applichiamo il metodo describe al nuovo DataFrame `lst_for_price_analysis` per avere una prima idea della distribuzione dei dati dopo il filtraggio.

In [ ]:
lst_for_price_analysis.describe()

La `latitudine`, come aspettato, è compresa tra 35.6 e 46.5, mentre la `longitudine` tra 9 e 18.5, che corrispondono alla posizione geografica dell'Italia. Per quanto riguarda le variabili `accomodates`, `bathrooms`, `bedrooms` e `beds`, si nota che sono presenti dei valori outliers.

Difatti il numero massimo di `accomodates` è 16, ancora accettabile, ma ben più alto dal valore del percentile 75 che equivale a 5. Per di più il numero massimo di `bathrooms` è 100, di `bedrooms` è 44 e di `beds` è 50, che sono valori molto elevati e potrebbero essere considerati outliers risultato di errori di immissione.

Questi valori si discostano significativamente dai dati, e possono influenzare negativamente la performance del modello di regressione. Per questo motivo, è necessario trattarli prima di procedere con la fase di modellazione.

Per quanto riguarda la variabile `price`, si nota che il prezzo massimo è 80000, mentre il prezzo del 75-esimo percentile è 168. Questo indica che ci sono dei valori di prezzo molto elevati che potrebbero anche essi essere considerati outliers.

Prima di andare a visualizzare con dei plot la distribuzione delle variabili, trattiamo i valori mancanti, visti nella sezione precedente. Visualizziamo comunque per ogni variabile il numero di valori mancanti:

In [ ]:
na_values = lst_for_price_analysis.isna().sum()
print(na_values)

Come possiamo vedere le variabili `bathrooms`, `bedrooms` e `beds` contengono un numero poco significativo di valori mancanti rispetto ai dati totali. Si suppone che questi valori mancanti siano dovuti a errori di immissione, siccome è impossibile avere un alloggio con 0 camere o 0 letti. Per quanto riguarda i bagni, è possibile che ci siano alloggi senza bagno, ma è più probabile che si tratti di errori di immissione. 

Per questi motivi:
- Per le variabili `bedrooms` e `beds`, si suppone che i valori mancanti siano dovuti a errori di immissione e verranno tolti dal dataset.
- Per la variabile `bathrooms`, si suppone che effettivamente ci siano alloggi senza bagno, quindi i valori mancanti verranno sostituiti con 0.

In [ ]:
lst_for_price_analysis = lst_for_price_analysis.dropna(subset=["bedrooms", "beds"])
lst_for_price_analysis["bathrooms"] = lst_for_price_analysis["bathrooms"].fillna(0)
na_values = lst_for_price_analysis.isna().sum()
print(na_values)

Ora non abbiamo più valori mancanti nelle variabili `bathrooms`, `bedrooms` e `beds`, e possiamo procedere con la visualizzazione della distribuzione delle variabili tramite dei plot.

Come primo plot, è possibile visualizzare la distribuzione della variabile `price` tramite un istogramma. In questo modo è possibile vedere se ci sono dei valori di prezzo molto elevati che potrebbero essere considerati outliers.

In [ ]:
lst_for_price_analysis["price"].plot.hist(
    bins=50,
    log=True,
)

Come si può vedere, la distribuzione della variabile `price` è molto sbilanciata, con la maggior parte dei valori concentrati tra 0 e 10000, con alcuni valori molto elevati che potrebbero essere considerati outliers. Decidiamo di togliere i valori estremi, in particolare sopra i 10000 e sotto i 5. Il grafico è in scala logaritmica, in modo tale che anche i valori più elevati siano visibili. Questa cosa può essere vista anche con il boxplot:

In [ ]:
lst_for_price_analysis["price"] = lst_for_price_analysis["price"].clip(upper=10000, lower=5)
lst_for_price_analysis["price"].plot.hist(
    bins=50,
    log=True,
)

Ora si può vedere che la distribuzione di price si concentra su due picchi principali, uno intorno a 5-2000 e l'altro intorno a 7000-1000. Questo potrebbe indicare che ci sono due categorie principali di alloggi, alcuni più economici e altri che potrebbero essere considerati di lusso.

Controlliamo ora il boxplot della variabile `price` per identificare visivamente la presenza di outliers e la distribuzione dei prezzi.

In [ ]:
lst_for_price_analysis["price"].plot.box()

Proviamo a creare un istogramma colorato in base al `room_type`, per vedere se ci sono differenze nella distribuzione dei prezzi in base al tipo di alloggio.

In [ ]:
# Proviamo a creare un istogramma colorato in base al `room_type`, per vedere se ci sono differenze nella distribuzione dei prezzi in base al tipo di alloggio.
room_types = lst_for_price_analysis["room_type"].values
plt.figure(figsize=(10, 6))
sns.histplot(data=lst_for_price_analysis, x="price", hue=room_types, bins=50, log_scale=True)
plt.title("Distribuzione dei prezzi per room_type")
plt.xlabel("Price (log scale)")
plt.ylabel("Count")
plt.legend(title="Room Type")
plt.show()

Generiamo ora degli istogrammi per le variabili `accomodates`, `bathrooms`, `bedrooms` e `beds` per visualizzare meglio la presenza di outliers.

In [ ]:
cols = ["accommodates", "bathrooms", "bedrooms", "beds"]

fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.ravel() # Appiattisce la matrice di assi in un array 1D per iterare più facilmente

for ax, col in zip(axes, cols):
    lst_for_price_analysis[col].dropna().plot.hist(
        bins=50,
        log=True,
        ax=ax,
    )
    ax.set_title(col.capitalize())
    ax.set_xlabel(col)
    ax.set_ylabel("Frequency")

plt.tight_layout()
plt.show()

Come si può vedere bathrooms, bedrooms e beds presentano dei valori molto elevati che sono il risultato di probabili errori di immissione. Per questo motivo questi valori saranno eliminati dal dataset:
- `bathrooms` < 20
- `bedrooms` < 22
- `beds` < 35

In [ ]:
lst_wout_outl_p_a = lst_for_price_analysis[
    (lst_for_price_analysis["bathrooms"] < 20) &
    (lst_for_price_analysis["bedrooms"] < 22) &
    (lst_for_price_analysis["beds"] < 35)
]

Adesso generiamo i boxplot per tutte le variabili per visualizzare il risulato del filtraggio degli outliers dati da palesi errori di immissione.

In [ ]:
cols = ["accommodates", "bathrooms", "bedrooms", "beds", "price"]

fig, axes = plt.subplots(len(cols) // 2 + len(cols) % 2, len(cols) // 2, figsize=(12, 10))
axes = axes.ravel() # Appiattisce la matrice di assi in un array 1D per iterare più facilmente

for ax, col in zip(axes, cols):
    lst_wout_outl_p_a[col].dropna().plot.box(
        ax=ax,
    )
    ax.set_title(col.capitalize())
    ax.set_xlabel(col)
    ax.set_ylabel("Frequency")

plt.tight_layout()
plt.show()

Ora proviamo a visualizzare la relazione tra `price` e le variabili `accomodates`, `bathrooms`, `bedrooms` e `beds` tramite dei scatter plot. In questo modo è possibile vedere se ci sono delle relazioni tra queste variabili e il prezzo, e se ci sono dei valori di prezzo molto elevati che potrebbero essere considerati outliers.

In [ ]:
cols = ["accommodates", "bathrooms", "bedrooms", "beds"]
colors = ["blue", "orange", "green", "red"]

sample = lst_wout_outl_p_a.sample(n=5000, random_state=7112004)

fig, axes = plt.subplots(len(cols) // 2 + len(cols) % 2, len(cols) // 2, figsize=(12, 10))
axes = axes.ravel()

for ax, col in zip(axes, cols):
    sample[["price", col]].plot.scatter(
        x=col,
        y="price",
        ax=ax,
        logy= True,
        c=colors[cols.index(col)]
    )
    ax.set_title(col.capitalize())
    ax.set_xlabel(col)
    ax.set_ylabel("Price")

plt.tight_layout()
plt.show()

In tutti e quattro i grafici si vede una tendenza positiva, anche se debole: all'aumentare del numero di `accomodates`, `bathrooms`, `bedrooms` e `beds`, aumenta anche il prezzo. Sicuramente variabili categoriche che abbiamo lasciato fuori da questa analisi, come `neighbourhood_cleansed`, `property_type`, `room_type` e `amenities`, possono avere anche esse un impatto sul prezzo, e potrebbero essere utili per migliorare la performance del modello di regressione.

Adesso analizziamo queste variabili. Estraiamo i valori unici per ogni variabile categorica:

Le ultime variabili da esplorare che possono essere utili per la predizione del prezzo sono le densità di letti, camere e bagni per ogni alloggio. Aggiungiamo quindi quattro nuove variabili al dataset:
- `beds_per_person` = `beds` / `accomodates`: il numero di letti per persona. Un valore più alto di questa variabile potrebbe indicare un alloggio più confortevole, e quindi potrebbe essere associato a un prezzo più elevato.
- `bedrooms_per_person` = `bedrooms` / `accomodates`: il numero di camere da letto per persona. Un valore più alto di questa variabile potrebbe indicare un alloggio più spazioso, e quindi potrebbe essere associato a un prezzo più elevato.
- `bathrooms_per_person` = `bathrooms` / `accomodates`: il numero di bagni per persona. Un valore più alto di questa variabile potrebbe indicare un alloggio più confortevole, e quindi potrebbe essere associato a un prezzo più elevato.
- `beds_per_bedroom` = `beds` / `bedrooms`: il numero di letti per camera da letto. Un valore più alto di questa variabile potrebbe indicare un alloggio più spazioso, e quindi potrebbe essere associato a un prezzo più elevato.

In [ ]:
lst_for_price_preproc = lst_wout_outl_p_a.copy()
lst_for_price_preproc["beds_per_person"] = lst_for_price_preproc["beds"] / lst_for_price_preproc["accommodates"]
lst_for_price_preproc["bedrooms_per_person"] = lst_for_price_preproc["bedrooms"] / lst_for_price_preproc["accommodates"]
lst_for_price_preproc["bathrooms_per_person"] = lst_for_price_preproc["bathrooms"] / lst_for_price_preproc["accommodates"]
lst_for_price_preproc["beds_per_bedroom"] = lst_for_price_preproc["beds"] / lst_for_price_preproc["bedrooms"]
print(lst_for_price_preproc)

Ora creiamo un grafico nello stile di quello superiore, per visualizzare se ci sono delle relazioni tra queste nuove variabili e il prezzo.

In [ ]:
cols = ["beds_per_person", "bedrooms_per_person", "bathrooms_per_person", "beds_per_bedroom"]
sample = lst_for_price_preproc.sample(n=7000, random_state=7112004)

fig, axes = plt.subplots(len(cols) // 2 + len(cols) % 2, len(cols) // 2, figsize=(12, 10))
axes = axes.ravel()

for ax, col in zip(axes, cols):
    sample[["price", col]].plot.scatter(
        x=col,
        y="price",
        ax=ax,
        logy= True,
        logx= True,
        c=colors[cols.index(col)]
    )
    ax.set_title(col.capitalize())
    ax.set_xlabel(col)
    ax.set_ylabel("Price")

plt.tight_layout()
plt.show()

Tutte e quattro le variabili non mostrano una relazione molto forte con il prezzo. Per una visualizzazione più chiara è stato utilizzato il logaritmo anche dell'asse x. Possono però essere effettuate le seguenti osservazioni:
- `beds_per_person` e `bedrooms_per_person`: si nota una lieve concentrazione dei prezzi più elevati in corrispondenza di valori prossimi a 1, ovvero listing con un letto o una camera per ospite. Tuttavia la dispersione è elevata, quindi il potere predittivo di queste feature è limitato.
- `bathrooms_per_person`: non emerge alcuna tendenza chiara. La distribuzione è sostanzialmente piatta lungo l'asse x, indicando una scarsa correlazione con il prezzo.
- `beds_per_bedroom`: si osserva una leggera tendenza inversa, infatti listing con più letti per camera (ipoteticamente dormitori o strutture condivise) tendono ad avere prezzi più bassi. Questa feature potrebbe quindi catturare indirettamente anche la tipologia di alloggio.

#### 2.3.b - Occupancy Regression

Obiettivo: esplorare `estimated_occupancy_l365d` e le sue relazioni con le feature disponibili, escludendo le colonne che creerebbero leakage: `price`, `estimated_revenue_l365d`, tutte le colonne `availability_*` e le colonne `number_of_reviews*`.

Sottosezioni:

- **A**: Panoramica target (distribuzione, missingness, statistica descrittiva).
- **B**: Correlazioni numeriche — elenco feature più correlate (positive/negative) con il target.
- **C**: Scatter / regplot per le top feature correlate.
- **D**: Analisi categorica: boxplot di `estimated_occupancy_l365d` per `room_type` e per i top quartieri.
- **E**: Missingness per feature rilevanti.

In [ ]:
# A Target overview: `estimated_occupancy_l365d`
t = listings['estimated_occupancy_l365d']
print('Count non-null:', t.count())
print('Missing:', t.isna().sum())
display(t.describe())
plt.figure(figsize=(8,4))
sns.histplot(t.dropna(), kde=True, bins=50)
plt.title('Distribution of estimated_occupancy_l365d')
plt.xlabel('estimated_occupancy_l365d')
plt.show()
plt.figure(figsize=(6,3))
sns.boxplot(x=t)
plt.title('Boxplot of estimated_occupancy_l365d')
plt.show()

### Come leggere i grafici

- Istogramma + KDE: mostra la forma della distribuzione (asimmetria, picchi, code).
- Boxplot: mette in evidenza mediana, IQR e outlier.
- Statistiche rapide: media, mediana, quartili, count e missing.

### Cosa emerge

- Distribuzione fortemente asimmetrica con massa vicino a 0 e lunga coda verso valori alti: molti annunci hanno poche prenotazioni mentre pochi sono molto popolari (hotel, appartamenti centrali).
- Mediana relativamente bassa: la maggioranza degli annunci presenta occupazione contenuta.
- Presenza di outlier (valori molto alti) che vanno verificati ma non sono necessariamente errori.
- Potrebbe essere utile considerare trasformazioni del target o modelli robusti se si cerca stabilità nelle previsioni.


In [ ]:
# B Correlazioni con il target
exclude_prefixes = ['availability_', 'number_of_reviews']
exclude_exact = {'price', 'estimated_revenue_l365d'}
num_cols = listings.select_dtypes(include=[np.number]).columns.tolist()
if 'estimated_occupancy_l365d' in num_cols:
    num_cols.remove('estimated_occupancy_l365d')
cols_for_corr = [c for c in num_cols if c not in exclude_exact and not any(c.startswith(p) for p in exclude_prefixes)]
print(f'Total numeric features considered for correlation: {len(cols_for_corr)}')
corr_with_target = listings[cols_for_corr + ['estimated_occupancy_l365d']].corr()['estimated_occupancy_l365d'].drop('estimated_occupancy_l365d')
corr_sorted = corr_with_target.sort_values(ascending=False)
display(corr_sorted.head(20))
display(corr_sorted.tail(20))

### Come leggere

- La tabella ordina le feature per correlazione con `estimated_occupancy_l365d` (dal più positivo al più negativo).
- Guardare sia il segno (positivo/negativo) sia la magnitudo: le feature con valore assoluto più alto sono le più rilevanti.
- Sono state escluse le colonne di possibile leakage: `price`, `estimated_revenue_l365d`, `availability_*` e `number_of_reviews*`.

### Cosa emerge

- Le metriche di recensione (es. `reviews_per_month`, `review_scores_*`) mostrano le associazioni più evidenti.
- Gli effetti osservati sono generalmente moderati: le correlazioni non sono enormi e possono essere influenzate da confondenti o da differenze fra città.
- Usare queste indicazioni per selezionare le candidate feature da approfondire con scatterplot e analisi stratificate.


In [ ]:
# C Scatter / regplot per le top feature correlate
corr_vals = corr_with_target.abs().sort_values(ascending=False)
top_feats = corr_vals.head(5).index.tolist()
print('Top features by absolute correlation:', top_feats)

n_cols = 3
n_rows = 2
fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, 10))
axes = axes.ravel()

for ax, f in zip(axes, top_feats):
    sns.regplot(x=listings[f], y=listings['estimated_occupancy_l365d'], scatter_kws={'alpha':0.35}, ax=ax)
    ax.set_title(f'{f} vs estimated_occupancy_l365d')
    ax.set_xlabel(f)
    ax.set_ylabel('estimated_occupancy_l365d')

for ax in axes[len(top_feats):]:
    ax.axis('off')

plt.tight_layout()
plt.show()

### Come leggere gli scatter/regplot

- Ogni grafico mostra la relazione tra una feature numerica e `estimated_occupancy_l365d`.
- I punti sparsi danno l'idea della variabilità reale dei dati, mentre la linea di regressione aiuta a capire se il legame è crescente, decrescente o quasi assente.
- Se la nuvola di punti è molto larga e la linea è quasi orizzontale, la feature è poco informativa da sola.
- La vista a scacchiera rende più facile confrontare i grafici e vedere subito quali relazioni sembrano più stabili.

### Cosa emerge

- Le feature più interessanti sono `reviews_per_month`, `latitude`, `longitude`, `review_scores_value` e `review_scores_accuracy`, ma il segnale non è sempre pulito.
- `reviews_per_month` sembra la più informativa, anche se la relazione è influenzata da alcuni valori molto estremi e da una forte concentrazione di punti nei valori bassi.
- Le variabili geografiche vanno lette con cautela: il dataset è aggregato per città, quindi `latitude` e `longitude` riflettono anche differenze tra mercati diversi.
- In generale, questi grafici servono soprattutto a individuare feature promettenti e a capire la forma della relazione, non a dire che una variabile è forte in senso assoluto.

In [ ]:
# D Analisi categorica (semplificata): room_type + neighbourhoods filtrati
min_count = 10

# Boxplot per `room_type`
plt.figure(figsize=(10,4))
order = listings['room_type'].value_counts().index
sns.boxplot(x='room_type', y='estimated_occupancy_l365d', data=listings, order=order)
plt.xticks(rotation=45)
plt.title('Occupancy by room_type (top categories)')
plt.show()

# Boxplot per `neighbourhood_cleansed` ma escludendo quartieri con pochi listing
counts = listings['neighbourhood_cleansed'].value_counts()
good_neigh = counts[counts >= min_count].index
print(f'Plotting neighbourhoods with >= {min_count} listings ({len(good_neigh)} neighbourhoods).')
top_to_plot = counts.loc[good_neigh].sort_values(ascending=False).head(15).index
if len(top_to_plot) > 0:
    plt.figure(figsize=(12,5))
    sns.boxplot(x='neighbourhood_cleansed', y='estimated_occupancy_l365d',
                data=listings[listings['neighbourhood_cleansed'].isin(top_to_plot)],
                order=top_to_plot)
    plt.xticks(rotation=45)
    plt.title(f'Occupancy by neighbourhood (min_count={min_count}) - top {len(top_to_plot)}')
    plt.show()

counts = listings['neighbourhood_cleansed'].value_counts()
filtered = counts[counts >= min_count].sort_values(ascending=False)
print(filtered.to_string())

### Come leggere i boxplot

- Ogni boxplot confronta la distribuzione di `estimated_occupancy_l365d` tra le categorie (es. `room_type` o `neighbourhood_cleansed`).
- La linea centrale è la mediana; il box mostra l'IQR (50% centrale). Baffi e punti fuori indicano variabilità e outlier.
- Attenzione alle categorie con pochi annunci: medie elevate possono dipendere da pochi valori estremi.
- In questa versione abbiamo applicato un filtro di supporto: vengono plottati solo i quartieri con almeno `min_count` listing (default 10). Controllare sempre il valore di `min_count` usato nella cella di codice.
- Controllare sempre la frequenza (count) per categoria insieme alla mediana prima di trarre conclusioni.

### Cosa emerge

- Alcuni quartieri presentano medie molto alte (es. Lenna, Pra' Secco, Ca' Brentelle): prima di considerare questi risultati, verificare il conteggio di listing per quei quartieri — se il conteggio è basso, la mediana o la media può essere fuorviante.
- `room_type` mostra differenze pratiche: intere case tendono ad avere mediane e variabilità maggiori rispetto a stanze condivise o private.
- Il filtro `min_count` riduce l'impatto dei quartieri con supporto scarso, rendendo le comparazioni più robuste; tuttavia potrebbe escludere nicchie reali se `min_count` è troppo alto.
- Questi grafici servono a individuare categorie e quartieri da approfondire con analisi stratificate o con filtri per numero di osservazioni; usare la stampa dei conteggi (ordinata per frequenza) per ispezionare i quartieri esclusi.


In [ ]:
# E Missingness summary per feature rilevanti
relevant = cols_for_corr.copy() if 'cols_for_corr' in globals() else []
relevant += ['estimated_occupancy_l365d'] if 'estimated_occupancy_l365d' in listings.columns else []
miss_pct = listings[relevant].isna().mean().sort_values(ascending=False)
print('Missingness percentage for relevant numeric features:')
display(miss_pct.head(20))
plt.figure(figsize=(6,8))
sns.heatmap(listings[relevant].isna().astype(int).sample(frac=0.2, random_state=42).T, cbar=False)
plt.title('Missingness heatmap (sampled rows)')
plt.show()

### Come leggere la missingness

- La tabella mostra la percentuale di valori mancanti per ciascuna feature: valori più alti indicano meno osservazioni utili per quella colonna.
- La heatmap campionata visualizza la struttura dei missing (colonne = variabili, righe = campione di record): strisce verticali indicano missing sparsi, blocchi orizzontali indicano assenze sistematiche su gruppi di righe.
- Confrontare sempre missingness e target: se il target è completo, le feature mancanti possono essere imputate o segnalate con indicatori di missing.
- Prestare attenzione a variabili derivate da recensioni (`review_scores_*`, `reviews_per_month`): spesso hanno missing non casuale (dipendono dall'assenza di recensioni).

### Cosa emerge

- Le feature legate alle recensioni mostrano missing non trascurabile: considerare imputazione (mediana) o l'uso di una variabile indicator per missingness.
- La heatmap suggerisce pattern sparsi piuttosto che blocchi grandi: i missing tendono a essere associati a singoli annunci, non a interi sottogruppi (ma verificare stratificando per città/quartiere).
- Regole pratiche consigliate: valutare l'eliminazione di colonne con percentuale di missing molto alta (es. >50%), oppure imputare con strategie stratificate; testare modelli con/senza le colonne imputate.
- Se i missing sono correlati a feature geografiche o categorie, preferire imputazione stratificata per evitare bias.


## 3. Preprocessing

#### 3.1 Preprocessing per Price Regression

Passiamo alla fase di preprocessing per il task di regressione su `price`. In questa fase, è necessario preparare i dati in modo che siano adatti per l'addestramento del modello di regressione. I passi che seguono sono:


*Scrivi passi quando li fai*

Procediamo con il creare un nuovo DataFrame `lst_for_price_preproc` a partire da `listings`, che contiene solo le feature necessarie per la predizione del prezzo. Effettuiamo il filtraggio delle feature come fatto nella sezione precedente. Lo ripetiamo qui per maggiore chiarezza:

In [ ]:
lst_for_price_preproc = listings.copy()

# Definizione delle feature da mantenere per il task di regressione su price
filtered_features = [
    "id",
    "city",
    "neighbourhood_cleansed",
    "latitude",
    "longitude",
    "property_type",
    "room_type",
    "accommodates",
    "bathrooms",
    "bedrooms",
    "beds",
    "amenities",
    "price"
]

# Filtraggio feature per il task di regressione su price
lst_for_price_preproc = lst_for_price_preproc[filtered_features]

lst_for_price_preproc = lst_for_price_preproc.rename(columns={"neighbourhood_cleansed": "neighbourhood"})

# Trattamento della variabile price per trasformarla in un valore numerico
lst_for_price_preproc["price"] = (
    lst_for_price_preproc["price"]
        .str.replace('$', '', regex=False)
        .str.replace(',', '', regex=False)
        .astype(float)
)

# Trattamento dei valori mancanti per le feature numeriche
lst_for_price_preproc = lst_for_price_preproc.dropna(subset=["bedrooms", "beds"])
lst_for_price_preproc["bathrooms"] = lst_for_price_preproc["bathrooms"].fillna(0)

# Filtraggio outliers dati da palesi errori di immissione
lst_for_price_preproc = lst_for_price_preproc[
    (lst_for_price_preproc["bathrooms"] < 20) &
    (lst_for_price_preproc["bedrooms"] < 22) &
    (lst_for_price_preproc["beds"] < 35)
]

lst_for_price_preproc["price"] = lst_for_price_preproc["price"].clip(upper=10000, lower=5)
lst_for_price_preproc["price"] = np.log1p(lst_for_price_preproc["price"])

lst_for_price_preproc["amenities"] = lst_for_price_preproc["amenities"].apply(ast.literal_eval)

# Creazione di nuove feature a partire da quelle esistenti
lst_for_price_preproc["beds_per_person"] = lst_for_price_preproc["beds"] / lst_for_price_preproc["accommodates"]
lst_for_price_preproc["bedrooms_per_person"] = lst_for_price_preproc["bedrooms"] / lst_for_price_preproc["accommodates"]
lst_for_price_preproc["bathrooms_per_person"] = lst_for_price_preproc["bathrooms"] / lst_for_price_preproc["accommodates"]
lst_for_price_preproc["beds_per_bedroom"] = lst_for_price_preproc["beds"] / lst_for_price_preproc["bedrooms"]

# Trattamento di inf e -inf derivanti da divisioni per zero o valori nulli
cols = ["beds_per_person", "bedrooms_per_person", "bathrooms_per_person", "beds_per_bedroom"]

lst_for_price_preproc[cols] = (
    lst_for_price_preproc[cols]
    .replace([np.inf, -np.inf], np.nan)
    .fillna(0)
)

Guardiamo ora il risultato del preprocessing:

In [ ]:
lst_for_price_preproc.head()

Andiamo ora a modificare le categorie di `property_type` in modo da avere solo i top 10 più frequenti, e gli altri valori raggruppati in una categoria "other". In questo modo, si riduce la dimensionalità della variabile categorica, mantenendo comunque le categorie più rappresentative per la predizione del prezzo.

In [ ]:
top_property_types = lst_for_price_preproc["property_type"].value_counts().sort_values(ascending=False).head(30).index
lst_for_price_preproc["property_type"] = lst_for_price_preproc["property_type"].apply(lambda x: x if x in top_property_types else "Other")
lst_for_price_preproc.head()

Creiamo una variabile neighbourhood_price_median che rappresenta la mediana dei prezzi per ogni quartiere, in modo da avere una variabile numerica che rappresenta il prezzo mediano per quartiere.

In [ ]:
lst_for_price_preproc['neighbourhood_price_median'] = lst_for_price_preproc.groupby('neighbourhood')['price'].transform('median')

Facciamo la stessa cosa per `neighbourhood`, mantenendo solo i top 50 quartieri più frequenti e raggruppando gli altri in "other". In questo modo, si riduce la dimensionalità della variabile categorica, mantenendo comunque i quartieri più rappresentativi per la predizione del prezzo.

In [ ]:
top_neighbourhoods = lst_for_price_preproc["neighbourhood"].value_counts().sort_values(ascending=False).head(30).index
lst_for_price_preproc["neighbourhood"] = lst_for_price_preproc["neighbourhood"].apply(lambda x: x if x in top_neighbourhoods else "Other")
lst_for_price_preproc.head()

Per le amenities, si è deciso di mantenere solo i 20 servizi più frequenti, e di raggruppare gli altri in una categoria "other". In questo modo, si riduce la dimensionalità della variabile categorica. Per ottenere un altra informazione utile, si è deciso di creare una nuova variabile numerica `n_amenities`, che rappresenta il numero di servizi offerti da ogni alloggio. Questa variabile potrebbe essere utile per la predizione del prezzo, in quanto un alloggio con più servizi potrebbe essere associato a un prezzo più elevato.
Facciamo diventare amenities un set in modo da non ripetere più volte lo stesso servizio.

In [ ]:
lst_for_price_preproc["n_amenities"] = lst_for_price_preproc["amenities"].apply(len)

top_amenities = lst_for_price_preproc["amenities"].explode().value_counts().sort_values(ascending=False).head(30).index
lst_for_price_preproc["amenities"] = lst_for_price_preproc["amenities"].apply(lambda x: list({a if a in top_amenities else "Other" for a in x}))

In [ ]:
lst_for_price_preproc.head()

Un altro dato utile da estrarre potrebbe essere il calcolo della distanza di ogni alloggio dal centro della città, in modo da avere una variabile numerica che rappresenta la posizione geografica dell'alloggio. Per fare ciò, bisogna calcolare la distanza tra le coordinate geografiche di ogni alloggio e le coordinate del centro della città. In questo modo, si ottiene una nuova variabile numerica `distance_from_city_center` che potrebbe essere utile per la predizione del prezzo, siccome gli alloggi più vicini al centro potrebbero essere associati a un prezzo più elevato.

Creiamo quindi un dizionario con le coordinate del centro di ogni città presente nel dataset, in modo da poter calcolare la distanza di ogni alloggio dal centro della città. I dati sono stati presi da un LLM, e sono approssimativi, ma dovrebbero essere sufficienti per il nostro scopo.

In [ ]:
city_centers = {
    "bergamo": [{"lat": 45.6983, "lon": 9.6773}],
    "bologna": [{"lat": 44.4949, "lon": 11.3426}],
    "firenze": [{"lat": 43.7696, "lon": 11.2558}],
    "milano":  [{"lat": 45.4654, "lon": 9.1859}],
    "napoli":  [{"lat": 40.8518, "lon": 14.2681}],
    "puglia": [
        {"city": "Bari",      "lat": 41.1171, "lon": 16.8719},
        {"city": "Foggia",    "lat": 41.4621, "lon": 15.5444},
        {"city": "Taranto",   "lat": 40.4644, "lon": 17.2470},
        {"city": "Lecce",     "lat": 40.3516, "lon": 18.1752},
        {"city": "Brindisi",  "lat": 40.6326, "lon": 17.9417},
        {"city": "Andria",    "lat": 41.2278, "lon": 16.2961},
        {"city": "Barletta",  "lat": 41.3197, "lon": 16.2820},
    ],
    "roma":    [{"lat": 41.9028, "lon": 12.4964}],
    "sicilia": [
        {"city": "Palermo",   "lat": 38.1157, "lon": 13.3615},
        {"city": "Catania",   "lat": 37.5079, "lon": 15.0830},
        {"city": "Messina",   "lat": 38.1938, "lon": 15.5540},
        {"city": "Siracusa",  "lat": 37.0755, "lon": 15.2866},
        {"city": "Agrigento", "lat": 37.3111, "lon": 13.5765},
        {"city": "Trapani",   "lat": 38.0176, "lon": 12.5365},
        {"city": "Ragusa",    "lat": 36.9282, "lon": 14.7256},
    ],
    "trentino": [
        {"city": "Trento",      "lat": 46.0748, "lon": 11.1217},
        {"city": "Bolzano",     "lat": 46.4983, "lon": 11.3548},
        {"city": "Rovereto",    "lat": 45.8912, "lon": 11.0396},
        {"city": "Merano",      "lat": 46.6713, "lon": 11.1594},
        {"city": "Bressanone",  "lat": 46.7154, "lon": 11.6565},
    ],
    "venezia": [{"lat": 45.4408, "lon": 12.3155}],
}

Per il centro delle regioni sono stati scelti i centri delle città più importanti. Ora creiamo una funzione `calculate_distances_from_city_center` che calcola la distanza tra le coordinate di ogni alloggio e le coordinate del centro della città. Siccome le regioni hanno più città, facciamo in modo che ritorni la lista delle distanze da ogni centro, in modo da poter prendere la distanza minima come distanza dell'alloggio dal centro della regione.

Siccome la terra è sferica, è necessario utilizzare la formula dell'haversine per calcolare la distanza tra due punti sulla superficie terrestre. L'idea della formula è quella di, dati due punti su una sfera, calcolare la lunghezza dell'arco che li collega passando per la sua superficie. 

Quindi:
1. Convertiamo le coordinate da gradi a radianti.
2. Calcoliamo le differenze di latitudine e longitudine.
3. Applichiamo la formula dell'haversine per ottenere la distanza in chilometri.

La formula dell'haversine è la seguente:

$$a = sin(\frac{dlat}{2})^2 + cos(lat1) * cos(lat2) * sin(\frac{dlon}{2})^2$$
$$c = 2 * atan2(\sqrt{a}, \sqrt{1−a})$$
$$d = R * c$$
Dove:
- `dlat` è la differenza di latitudine in radianti.
- `dlon` è la differenza di longitudine in radianti.
- `R` è il raggio della Terra (circa 6371 km).

Otteniamo alla fine $d$, che rappresenta la distanza in chilometri tra l'alloggio e il centro della città.

In [ ]:
def haversine(lat1, lon1, lat2, lon2):
    R = 6371000  # raggio della Terra in metri
    lat1, lon1, lat2, lon2 = np.radians([lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat / 2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2)**2
    return R * 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))

In [ ]:
def calculate_distances_from_city_center(city, lat, lon, city_centers=city_centers):
    centers = city_centers.get(city, [])
    if not centers:
        return None
    distances = [haversine(lat, lon, c["lat"], c["lon"]) for c in centers]
    return distances if distances else None

Applichiamo ora il metodo apply alle colonne `city`, `latitude`, e `longitude` per calcolare la distanza di ogni alloggio dal centro della città, e creiamo la nuova colonna.

In [ ]:
lst_for_price_preproc["distance_from_city_center"] = lst_for_price_preproc.apply(
    lambda row: min(calculate_distances_from_city_center(row["city"], row["latitude"], row["longitude"])),
    axis=1
)

In [ ]:
lst_for_price_preproc.head()

Tuttavia potrebbero esserci anche degli altri punti di interesse, come ad esempio i punti di interesse turistico, che potrebbero essere utili per la predizione del prezzo. Per questo motivo, si potrebbe considerare di calcolare anche la distanza di ogni alloggio da questi punti di interesse, e creare delle nuove variabili numeriche che rappresentano queste distanze. Estraiamo sempre da un LLM le coordinate di alcuni punti di interesse turistico per le città e le regioni presenti nel dataset:

In [ ]:
poi_by_city = {
    "bergamo": [
        {"lat": 45.7040, "lon": 9.6623}, # Campanone
        {"lat": 45.7035, "lon": 9.6622}, # Cappella Colleoni
        {"lat": 45.7035, "lon": 9.6627}, # Cattedrale di Bergamo
        {"lat": 45.7034, "lon": 9.6623}, # Basilica di Santa Maria Maggiore
        {"lat": 45.7038, "lon": 9.6627}, # Palazzo della Ragione
        {"lat": 45.7041, "lon": 9.6630}, # Fontana Contarini
        {"lat": 45.7041, "lon": 9.6578}, # Casematte di San Giovanni
    ],
    "bologna": [
        {"lat": 44.4942, "lon": 11.3467}, # Le Due Torri (Asinelli e Garisenda)
        {"lat": 44.4938, "lon": 11.3431}, # Piazza Maggiore
        {"lat": 44.4919, "lon": 11.3433}, # Palazzo dell'Archiginnasio
        {"lat": 44.4911, "lon": 11.3436}, # Portici di Bologna
        {"lat": 44.4923, "lon": 11.3483}, # Piazza Santo Stefano
        {"lat": 44.4937, "lon": 11.3422}, # Torre dell'Orologio
        {"lat": 44.4896, "lon": 11.3440}, # Basilica di San Domenico
    ],
    "firenze": [
        {"lat": 43.7693, "lon": 11.2562}, # Palazzo Vecchio
        {"lat": 43.7731, "lon": 11.2560}, # Cattedrale di Santa Maria del Fiore
        {"lat": 43.7697, "lon": 11.2556}, # Piazza della Signoria
        {"lat": 43.7686, "lon": 11.2623}, # Basilica di Santa Croce
        {"lat": 43.7629, "lon": 11.2651}, # Piazzale Michelangelo
        {"lat": 43.7729, "lon": 11.2558}, # Campanile di Giotto
        {"lat": 43.7692, "lon": 11.2555}, # Loggia dei Lanzi
    ],
    "milano": [
        {"lat": 45.4641, "lon": 9.1919}, # Duomo di Milano
        {"lat": 45.4642, "lon": 9.1897}, # Piazza del Duomo
        {"lat": 45.4705, "lon": 9.1793}, # Castello Sforzesco
        {"lat": 45.4658, "lon": 9.1899}, # Galleria Vittorio Emanuele II
        {"lat": 45.4660, "lon": 9.1710}, # Basilica di Santa Maria delle Grazie
        {"lat": 45.4864, "lon": 9.1826}, # Torre Arcobaleno
        {"lat": 45.4692, "lon": 9.1809}, # Piazza Castello (Fontana)
    ],
    "napoli": [
        {"lat": 40.8651, "lon": 14.2474}, # Catacombe di San Gennaro
        {"lat": 40.8373, "lon": 14.2455}, # Napoli Sotterranea
        {"lat": 40.8362, "lon": 14.2494}, # Palazzo Reale di Napoli
        {"lat": 40.8536, "lon": 14.2505}, # Museo Archeologico Nazionale
        {"lat": 40.8385, "lon": 14.2527}, # Castel Nuovo (Maschio Angioino)
        {"lat": 40.8358, "lon": 14.2486}, # Piazza del Plebiscito
        {"lat": 40.8436, "lon": 14.2408}, # Certosa e Museo di San Martino
        {"lat": 40.8595, "lon": 14.2485}, # Catacombe di San Gaudioso
    ],
    "puglia": [
        {"lat": 41.1303, "lon": 16.8701}, # Basilica di San Nicola (Bari)
        {"lat": 41.1286, "lon": 16.8688}, # Cattedrale di San Sabino (Bari)
        {"lat": 41.1279, "lon": 16.8664}, # Castello Svevo di Bari
        {"lat": 40.8759, "lon": 17.1480}, # Grotte di Castellana
        {"lat": 41.0848, "lon": 16.2709}, # Castel del Monte
    ],
    "roma": [
        {"lat": 41.8921, "lon": 12.4864}, # Foro Romano
        {"lat": 41.9009, "lon": 12.4833}, # Fontana di Trevi
        {"lat": 41.8986, "lon": 12.4769}, # Pantheon
        {"lat": 41.8902, "lon": 12.4922}, # Colosseo
        {"lat": 41.8992, "lon": 12.4731}, # Piazza Navona
        {"lat": 41.9107, "lon": 12.4764}, # Piazza del Popolo
        {"lat": 41.8956, "lon": 12.4722}, # Campo de' Fiori
        {"lat": 41.9060, "lon": 12.4828}, # Scalinata di Trinità dei Monti
    ],
    "sicilia": [
        {"lat": 38.1132, "lon": 13.3530}, # Cattedrale di Palermo
        {"lat": 38.1109, "lon": 13.3517}, # Palazzo dei Normanni
        {"lat": 37.5025, "lon": 15.0871}, # Fontana dell'Elefante (Catania)
        {"lat": 37.5024, "lon": 15.0877}, # Basilica di Sant'Agata (Catania)
        {"lat": 37.5024, "lon": 15.0906}, # Palazzo Biscari (Catania)
        {"lat": 37.5090, "lon": 15.1025}, # Museo Sbarco in Sicilia 1943
        {"lat": 37.5022, "lon": 15.0877}, # Terme Achilliane (Catania)
    ],
    "trentino": [
        {"lat": 46.0715, "lon": 11.1271}, # Castello del Buonconsiglio
        {"lat": 46.0673, "lon": 11.1215}, # Piazza del Duomo (Trento)
        {"lat": 46.0695, "lon": 11.1214}, # Torre Mirana
        {"lat": 46.4983, "lon": 11.3548}, # Piazza Walther (Bolzano)
        {"lat": 46.5011, "lon": 11.3570}, # Museo di Scienze Naturali Alto Adige
        {"lat": 46.4828, "lon": 11.1322}, # Cascata di Tret
    ],
    "venezia": [
        {"lat": 45.4346, "lon": 12.3397}, # Basilica di San Marco
        {"lat": 45.4337, "lon": 12.3404}, # Palazzo Ducale
        {"lat": 45.4380, "lon": 12.3359}, # Ponte di Rialto
        {"lat": 45.4340, "lon": 12.3409}, # Ponte dei Sospiri
        {"lat": 45.4342, "lon": 12.3385}, # Piazza San Marco
        {"lat": 45.4340, "lon": 12.3390}, # Campanile di San Marco
        {"lat": 45.4348, "lon": 12.3368}, # Gondola Ride Experience
    ],
}

Da queste coordinate si possono ricavare delle nuove variabili numeriche che rappresentano la distanza di ogni alloggio da il punto di interesse più vicino, utilizzando la stessa funzione `calculate_distances_from_city_center` che abbiamo creato prima, ma adattandola per calcolare la distanza da un punto di interesse invece che dal centro della città.

Altre feature interessanti da estrarre potrebbero essere il numero di punti di interesse a meno di 250m, 500m, 1km, 2km, 5km. In questo modo, si ottengono nuove variabili numeriche che rappresentano la vicinanza di ogni alloggio a punti di interesse turistico, che potrebbero essere utili per la predizione del prezzo, in quanto gli alloggi più vicini a punti di interesse potrebbero essere associati a un prezzo più elevato. Creiamo quindi una nuova funzione `calculate_n_poi` che calcola il numero di punti di interesse a distanze variabili.

In [ ]:
def calculate_n_poi(city, lat, lon, thresh):
    distances = calculate_distances_from_city_center(city, lat, lon)
    if not distances:
        return None
    count = len([distance for distance in distances if distance <= thresh])
    return count

In [ ]:
distances_from_poi = lst_for_price_preproc.apply(
    lambda row: calculate_distances_from_city_center(row["city"], row["latitude"], row["longitude"], city_centers=poi_by_city),
    axis=1
)

print(distances_from_poi.head())

lst_for_price_preproc["distance_from_poi"] = distances_from_poi.apply(lambda dists: min(dists))

n_poi_250 = distances_from_poi.apply(lambda dists: sum(1 for d in dists if d <= 250))

n_poi_500 = distances_from_poi.apply(lambda dists: sum(1 for d in dists if d <= 500))

n_poi_1000 = distances_from_poi.apply(lambda dists: sum(1 for d in dists if d <= 1000))

n_poi_2000 = distances_from_poi.apply(lambda dists: sum(1 for d in dists if d <= 2000))

n_poi_5000 = distances_from_poi.apply(lambda dists: sum(1 for d in dists if d <= 5000))

Queste variabili sono fortemente correlate tra loro, il che potrebbe causare problemi di multicollinearità nel modello di regressione. Per questo motivo si potrebbe creare una nuova variabile che rappresenta la densità di punti di interesse, calcolata come media pesata del numero di punti di interesse a diverse distanze, in modo da ridurre la dimensionalità del dataset e migliorare la performance del modello di regressione. La formula per calcolare la densità di punti di interesse potrebbe essere la seguente:

$$poi\_density = n\_poi\_250m * w1 + n\_poi\_500m * w2 + n\_poi\_1km * w3 + n\_poi\_2km * w4 + n\_poi\_5km * w5$$
Dove `w1`, `w2`, `w3`, `w4`, e `w5` sono i pesi che rappresentano l'importanza relativa dei punti di interesse a diverse distanze.

Assegnamo quindi dei pesi decrescenti ai punti di interesse a distanze , la cui somma è 1, ovvero:
- `w1` = 0.25 (peso per i punti di interesse a 250 metri)
- `w2` = 0.25 (peso per i punti di interesse a 500 metri)
- `w3` = 0.25 (peso per i punti di interesse a 1 chilometro)
- `w4` = 0.15 (peso per i punti di interesse a 2 chilometri)
- `w5` = 0.1 (peso per i punti di interesse a 5 chilometri)

In [ ]:
lst_for_price_preproc["poi_density"] = n_poi_250 * 0.25 + n_poi_500 * 0.25 + n_poi_1000 * 0.25 + n_poi_2000 * 0.15 + n_poi_5000 * 0.1

In [ ]:
lst_for_price_preproc.head()

Procediamo ora con il preprocessing delle variabili categoriche `neighbourhood_cleansed`, `property_type`, `room_type` e `amenities`. Per fare ciò, è necessario trasformare queste variabili in nuove variabili binarie, in modo da poterle utilizzare per l'addestramento del modello di regressione. Usiamo i metodi di scikit-learn `OneHotEncoder` e `MultiLabelBinarizer` per trasformare le variabili categoriche in variabili binarie.

Tuttavia è necessario creare una classe custom MLBTransformer che utilizza `MultiLabelBinarizer` per trasformare la variabile `amenities`, in quanto questa variabile contiene una lista di servizi per ogni record, e non è possibile utilizzare direttamente MultiLabelBinarizer per trasformarla in variabili binarie.

Questa classe custom `MLBTransformer` implementa i metodi `fit`, `transform` e `fit_transform` per adattarsi alle interfacce BaseEstimator e TransformerMixin di scikit-learn. Il metodo `fit` adatta il `MultiLabelBinarizer` ai dati, il metodo `transform` trasforma i dati in un array binario, e il metodo `fit_transform` combina entrambe le operazioni. Inoltre si vuole mantenere i nomi delle feature originali, quindi si aggiunge un metodo `get_feature_names_out` che restituisce i nomi delle feature trasformate (che in MultiLabelBinarizer sono nell'attributo `classes_`).

In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.preprocessing import OneHotEncoder, MultiLabelBinarizer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split

class MLBTransformer(BaseEstimator, TransformerMixin):
    def __init__(self):
        self.mlb = MultiLabelBinarizer(sparse_output=False)

    def fit(self, X, y=None):
        self.mlb.fit(X)
        return self

    def transform(self, X):
        return self.mlb.transform(X)

    def get_feature_names_out(self, input_features=None):
        return np.array([c for c in self.mlb.classes_])

In [ ]:
def preprocess_data(listings, cols_to_drop, seed=7112004):
    categorical_cols = ["city", "property_type", "room_type", "neighbourhood"]
    amenities_col = "amenities"

    ohe = OneHotEncoder(sparse_output=False, drop="first")

    preprocessor = ColumnTransformer(
        transformers=[
            ("cat", ohe, categorical_cols),
            ("amenities", MLBTransformer(), amenities_col)
        ],
        remainder="passthrough"
    )

    X = listings.drop(columns=["id", "price"] + cols_to_drop)
    y = listings["price"]

    X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=1/3, random_state=seed)
    X_train_processed = preprocessor.fit_transform(X_train)
    X_val_processed = preprocessor.fit_transform(X_val)
    cols = X_train.columns.tolist() + preprocessor.get_feature_names_out().tolist()
    return X_train_processed, X_val_processed, y_train, y_val, cols, preprocessor


In [ ]:
X_train, X_val, y_train, y_val, cols, preprocessor = preprocess_data(lst_for_price_preproc, cols_to_drop=["latitude", "longitude"])

## 4. Addestramento e Valutazione dei Modelli

### 4.1 Price Regression

Partiamo ora con l'addestramento e la valutazione dei modelli per il task di regressione su `price`. In questa fase, è necessario utilizzare il dataset preprocessato per addestrare un modello di regressione che possa predire il prezzo degli alloggi sulla base delle feature disponibili. I passi che seguono sono:

**Scrivi passi quando li fai**

#### Valutazione dei modelli

Per valutare la performance del modello di regressione, è necessario utilizzare delle metriche di valutazione appropriate. Le metriche più comuni per la regressione sono:
- **MSE (Mean Squared Error)**: misura l'errore medio quadratico tra le predizioni del modello e i valori reali. Un valore più basso indica una migliore performance.
- **Relative Error (RE)**: misura l'errore relativo tra le predizioni del modello e i valori reali. Un valore più basso indica una migliore performance.
- **R² (R-squared)**: misura la proporzione della varianza nei dati che è spiegata dal modello. Un valore più alto indica una migliore performance, con 1 che rappresenta una perfetta predizione.

Definiamo le funzioni per calcolare queste metriche di valutazione:

In [ ]:
from sklearn.metrics import mean_squared_error

def relative_error(y_true, y_pred):
    return np.mean(np.abs((y_true - y_pred) / y_true))

def print_eval(X, y, model):
    print("   Mean squared error: {:.5}".format(mean_squared_error(model.predict(X), y)))
    print("       Relative error: {:.5%}".format(relative_error(model.predict(X), y)))
    print("R-squared coefficient: {:.5}".format(model.score(X, y)))

#### Regressione Lasso

*Scrivi*

In [ ]:
# Regressione Lasso

from sklearn.linear_model import Lasso
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import StandardScaler

model = Pipeline([
    ("scale", StandardScaler(with_mean=False)),
    ("regr", Lasso())
])

grid = {
    "regr__alpha": [0.0005, 0.001, 0.005]
}

# cols = ["rank_test_score","mean_test_score","mean_train_score","params"]
gs = GridSearchCV(model, param_grid=grid, cv=5)
gs.fit(X_train, y_train)

pd.DataFrame(gs.cv_results_).sort_values("mean_test_score", ascending=False)

In [ ]:
listings.info()

Come si può vedere il modello di regressione Lasso ha una performance piuttosto bassa, con un $R^2$ di 0.176. Questo è dovuto al fatto che abbiamo costruito un modello lineare, e quindi potrebbe non essere in grado di catturare le relazioni non lineari tra le feature e il prezzo.

In ogni caso possiamo guardare i coefficienti del modello a 0 per capire quali feature sono state considerate più importanti per la predizione del prezzo. Questo perchè la regolarizzazione L1 tende a azzerare i coefficienti delle feature meno importanti. Guardiamo i coefficienti a 0 con alpha=0.1:

In [ ]:
model = Pipeline([
    ("scale",  StandardScaler(with_mean=False)),
    ("regr", Lasso(alpha=0.0005))
])
model.fit(X_train, y_train);

In [ ]:
len(model.named_steps["regr"].coef_)

In [ ]:
lasso = pd.Series(model.named_steps["regr"].coef_, preprocessor.get_feature_names_out())
n_cols = lasso.count()
n_cols_zero = lasso[lasso != 0].count()
print(f"Numero di feature con coefficiente diverso da zero: {n_cols_zero} / {n_cols}")
print("Top feature più importanti (in valore assoluto):")
for feature, coef in lasso[lasso != 0].abs().sort_values(ascending=False).items():
    print(f"{feature}: {coef:.4f}")
print("Feature con coefficiente zero:")
for feature, coef in lasso[lasso == 0].items():
    print(f"{feature}: {coef:.4f}")

In [ ]:
print("Valutazione su training set:")
print_eval(X_train, y_train, model)
print("\nValutazione su validation set:")
print_eval(X_val, y_val, model)

I pesi che il modello di regressione Lasso ha assegnato alle feature Sono molto alti. Questo è un segnale che il modelo non riesce a catturare le relazioni tra le feature e il prezzo.

Probabilemente è necessario utilizzare un modello più complesso, come ad esempio un modello di regressione non lineare, o un modello di regressione con interazioni tra le feature, per migliorare la performance del modello di regressione.

Proviamo ora ad utilizzare un modello di regressione Ridge con feature polinomiali di diverso grado, per vedere se è in grado di catturare meglio le relazioni tra le feature e il prezzo, e quindi migliorare la performance del modello di regressione.

In [ ]:
# from sklearn.linear_model import Ridge
# from sklearn.preprocessing import PolynomialFeatures

# model = Pipeline([
#     ('poly', PolynomialFeatures(degree=2, interaction_only=True, include_bias=False)),
#     ('scaler', StandardScaler()),
#     ('ridge', Ridge())
# ])

# grid = {
#     'ridge__alpha': [0.1, 1, 10]
# }

# gs = GridSearchCV(model, param_grid=grid, cv=3)
# print(X_train)
# gs.fit(X_train, y_train)

# pd.DataFrame(gs.cv_results_).sort_values("mean_test_score", ascending=False)